# Setup lattice `.k` file and run

This notebook generates the lattice deck, then creates a system deck from `lattice_nose_cone_impact_test.k` and runs LS-DYNA.


In [1]:
from pathlib import Path
import shutil
import pandas as pd
from ansys.dyna.core import Deck, keywords as kwd

from utils import run_dyna, get_nodesets_lattice


In [ ]:
project_root = Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd().resolve()
dyna_folder_path = project_root / "DYNA_files"

pre_k_file_name = "lattice_nose_cone_pre.k" 
# Replace with original_nose_cone_star3_impact_test.k if you want to simulate the original nose cone instead of the lattice nose cone.
pre_k_file = dyna_folder_path / pre_k_file_name
generated_lattice_k_file = dyna_folder_path / "lattice_nose_cone_full.k"
system_template_file = dyna_folder_path / "lattice_nose_cone_impact_test.k"
system_k_file = dyna_folder_path / "lattice_nose_cone_impact_test_full.k"
run_file_folder = dyna_folder_path / "head_impact_test"

SOLVER = r"C:\Program Files\LS-DYNA Suite R16.1 Student\lsdyna\ls-dyna_smp_d_R16.1_180-gd50332dbe5_winx64_ifort190_sse2_studentversion.exe"
NCPU = 4
MEMORY = 200

PART_TITLE = "Lattice_nosecone"
PART_ID = 1
SECTION_ID = 1
MATERIAL_ID = 1
NODE_SET_ID = 1

SECTION_ELFORM = 10
MASS_DENSITY = 1.130e-06
YOUNG_MODULUS = 0.85
POISSON_RATIO = 0.35
YIELD_STRESS = 0.020
TANGENT_MODULUS = 0.013
FAILURE_STRAIN = 0.0

INITIAL_VELOCITY_Y = 10.0

FIXED_BOX = {
    "boxid": 1,
    "xmn": -100.0,
    "xmx": 100.0,
    "ymn": -5.0,
    "ymx": 0.5,
    "zmn": -100.0,
    "zmx": 100.0,
}

pre_k_file, generated_lattice_k_file, system_template_file


## Create the lattice nose cone deck


In [ ]:
def create_lattice_deck() -> Deck:
    deck = Deck()
    deck.title = PART_TITLE

    section = kwd.SectionSolid(secid=SECTION_ID)
    section.elform = SECTION_ELFORM

    material = kwd.Mat024(mid=MATERIAL_ID)
    material.ro = MASS_DENSITY
    material.e = YOUNG_MODULUS
    material.pr = POISSON_RATIO
    material.sigy = YIELD_STRESS
    material.etan = TANGENT_MODULUS
    material.fail = FAILURE_STRAIN

    part = kwd.Part()
    part.parts = pd.DataFrame(
        {
            "pid": [PART_ID],
            "mid": [material.mid],
            "secid": [section.secid],
            "title": [PART_TITLE],
        }
    )

    node_ids = get_nodesets_lattice(str(pre_k_file))["fixed_constraint"]

    fixed_nodes = kwd.SetNodeList(
        sid=NODE_SET_ID,
        nodes=node_ids,
        title="lattice fixed nodes",
    )

    initial_velocity = kwd.InitialVelocityGeneration()
    initial_velocity.id = PART_ID
    initial_velocity.styp = 2
    initial_velocity.vy = INITIAL_VELOCITY_Y

    include_path = kwd.IncludePath()
    include_path.path = str(dyna_folder_path)

    include_file = kwd.Include(filename=pre_k_file_name)

    deck.extend(
        [
            part,
            material,
            section,
            fixed_nodes,
            initial_velocity,
            include_path,
            include_file,
        ]
    )
    return deck


def write_lattice_deck(target_file: Path) -> Path:
    deck = create_lattice_deck()
    target_file.parent.mkdir(parents=True, exist_ok=True)
    deck.export_file(str(target_file))
    return target_file


In [ ]:
lattice_output = write_lattice_deck(generated_lattice_k_file)
print(f"Generated lattice deck: {lattice_output}")


## Create system deck

Use `lattice_nose_cone_impact_test.k` as the validated template for contact, control, and include-transform setup. Only replace the lattice include target and write a runnable system deck.


In [ ]:
deck = Deck()
deck.import_file(str(system_template_file))

include_path = kwd.IncludePath()
include_path.path = str(dyna_folder_path)

cards = deck.keywords
deck = Deck()
deck.title = "Lattice Nose Cone Impact Test"
deck.extend([include_path, *cards])

system_k_file.parent.mkdir(parents=True, exist_ok=True)
deck.export_file(str(system_k_file))

## Run the impact simulation

Run the generated system deck after regenerating the lattice include.


In [ ]:
run_dyna(
    ls_solver=SOLVER,
    k_file=str(system_k_file),
    run_folder=str(run_file_folder),
    ncpu=NCPU,
    memory=MEMORY,
)
